# Concept Drift Detection with ADWIN

This notebook visualizes what happens when user behavior shifts mid-stream
and shows how ADWIN detects the change — while a batch model would silently degrade.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.producer.event_generator import generate_events
from src.drift.detector import ADWIN, FeatureDriftMonitor
from src.consumer.feature_pipeline import SlidingWindowAggregator

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Simulate the stream and track CTR over time

In [ ]:
N = 10_000
DRIFT_AT = 0.5

events = list(generate_events(n_events=N, drift_at=DRIFT_AT))

# Track CTR for young_adult + eletrônicos over time (rolling window)
window = 200
target_segment = 'young_adult'
target_category = 'eletrônicos'

relevant = [
    int(e.event_type == 'click')
    for e in events
    if e.user_segment == target_segment and e.category == target_category
]

rolling_ctr = [
    np.mean(relevant[max(0, i-window):i])
    for i in range(window, len(relevant))
]
x = np.arange(len(rolling_ctr))

print(f"Total events: {N:,}")
print(f"Relevant events (young_adult + eletrônicos): {len(relevant):,}")
print(f"Phase 1 CTR: {np.mean(relevant[:len(relevant)//2]):.3f}")
print(f"Phase 2 CTR: {np.mean(relevant[len(relevant)//2:]):.3f}")

## 2. Run ADWIN on the stream and record detections

In [ ]:
detector = ADWIN(delta=0.002)
adwin_means = []
detections = []

for i, value in enumerate(relevant):
    drifted = detector.update(float(value))
    adwin_means.append(detector.mean)
    if drifted:
        detections.append(i)

print(f"ADWIN detections: {len(detections)}")
if detections:
    drift_point = int(len(relevant) * DRIFT_AT)
    first_detect = detections[0]
    delay = first_detect - drift_point
    print(f"True drift at sample: {drift_point}")
    print(f"First detection at:   {first_detect}")
    print(f"Detection delay:      {delay} samples")

## 3. Visualize: raw CTR + ADWIN detections + batch model degradation

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=False)

# Panel 1: Rolling CTR with drift marker
ax = axes[0]
ax.plot(rolling_ctr, color='#3498db', linewidth=1.5, label='Rolling CTR (w=200)')
drift_idx_rolling = int(len(rolling_ctr) * DRIFT_AT)
ax.axvline(drift_idx_rolling, color='#e74c3c', linestyle='--', linewidth=2, label='True drift point')
for d in detections:
    if d >= window:
        ax.axvline(d - window, color='#2ecc71', linestyle=':', alpha=0.7)
ax.set_ylabel('CTR')
ax.set_title(f'Rolling CTR — young_adult × eletrônicos\nCTR drops from ~0.15 to ~0.05 after drift', fontsize=11)
ax.legend()

# Panel 2: ADWIN running mean
ax = axes[1]
ax.plot(adwin_means, color='#9b59b6', linewidth=1.5, label='ADWIN running mean')
ax.axvline(int(len(relevant) * DRIFT_AT), color='#e74c3c', linestyle='--', linewidth=2)
for d in detections:
    ax.axvline(d, color='#2ecc71', linestyle=':', linewidth=1.5, alpha=0.8,
               label='Drift detected' if d == detections[0] else '')
ax.set_ylabel('ADWIN mean')
ax.set_title('ADWIN Adaptive Window Mean — reacts to distribution shift', fontsize=11)
ax.legend()

# Panel 3: Simulated batch model accuracy (fixed at phase-1 pattern, degrades in phase 2)
ax = axes[2]
phase1_ctr = np.mean(relevant[:len(relevant)//2])
batch_accuracy = [
    1.0 - abs(v - phase1_ctr) for v in rolling_ctr
]
ax.plot(batch_accuracy, color='#e74c3c', linewidth=1.5, label='Batch model accuracy (proxy)')
ax.fill_between(range(len(batch_accuracy)), batch_accuracy, alpha=0.2, color='#e74c3c')
ax.axvline(drift_idx_rolling, color='#e74c3c', linestyle='--', linewidth=2)
ax.set_xlabel('Event index (rolling window)')
ax.set_ylabel('Accuracy (proxy)')
ax.set_title('Batch model silently degrades after drift — no alert, no retraining', fontsize=11)
ax.legend()

plt.suptitle('Concept Drift: User Preference Shift Detected by ADWIN', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/01_drift_detection.png', bbox_inches='tight')
plt.show()

## 4. Detection delay vs. delta parameter

In [ ]:
deltas = [0.5, 0.1, 0.01, 0.002, 0.0002]
results = []

for delta in deltas:
    det = ADWIN(delta=delta)
    first_detection = None
    false_positives = 0
    drift_point = int(len(relevant) * DRIFT_AT)
    
    for i, value in enumerate(relevant):
        if det.update(float(value)):
            if i < drift_point:
                false_positives += 1
            elif first_detection is None:
                first_detection = i
    
    delay = (first_detection - drift_point) if first_detection else None
    results.append({'delta': delta, 'delay': delay, 'fp': false_positives})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
delays = [r['delay'] or 0 for r in results]
ax.semilogx([r['delta'] for r in results], delays, 'o-', color='#3498db', linewidth=2, markersize=8)
ax.set_xlabel('delta (ADWIN sensitivity)')
ax.set_ylabel('Detection delay (samples)')
ax.set_title('Smaller delta → faster detection\n(but more false positives)')

ax = axes[1]
fps = [r['fp'] for r in results]
ax.semilogx([r['delta'] for r in results], fps, 'o-', color='#e74c3c', linewidth=2, markersize=8)
ax.set_xlabel('delta (ADWIN sensitivity)')
ax.set_ylabel('False positives (before drift)')
ax.set_title('Larger delta → fewer false positives\n(but slower detection)')

plt.suptitle('ADWIN delta Trade-off: Speed vs. False Positive Rate', fontsize=12)
plt.tight_layout()
plt.savefig('figures/02_adwin_tradeoff.png', bbox_inches='tight')
plt.show()

print('\ndelta | delay | FP')
for r in results:
    print(f"{r['delta']:.4f} | {r['delay']} | {r['fp']}")